In [2]:
import json
import pandas as pd

In [3]:
import requests
import pandas as pd
import time

# =========================================================
# 1. FETCH THE COMPLETE HIERARCHY (Companies + Facilities)
# =========================================================
api_key = "e7149a636b4942e8fc7fe6c3d2fbb036"
url = "https://agsi.gie.eu/api/about?show=listing"

headers = {"x-key": api_key}
response = requests.get(url, headers=headers)

if response.status_code != 200:
    print("Failed:", response.status_code)
else:
    data = response.json()

# =========================================================
# 2. EXTRACT ALL FACILITIES WITH THEIR PARENT COMPANY
# =========================================================
all_facilities = []

for company in data:
    company_eic = company.get("eic")
    company_name = company.get("name")
    country = company.get("country")
    
    for facility in company.get("facilities", []):
        all_facilities.append({
            "country": country,
            "company_eic": company_eic,
            "company_name": company_name,
            "facility_name": facility["name"],
            "facility_eic": facility["eic"],
            "facility_type": facility.get("type"),
            "operational_start": facility.get("operational_start_date"),
            "operational_end": facility.get("operational_end_date")
        })

# =========================================================
# 3. SAVE TO CSV FOR FUTURE USE
# =========================================================
facilities_df = pd.DataFrame(all_facilities)
facilities_df.to_csv("europe_gas_storage_facilities.csv", index=False)
print(f"Found {len(facilities_df)} facilities")
print(facilities_df.head())

Found 128 facilities
  country       company_eic                company_name  \
0      AT  25X-GSALLC-----E                     GSA LLC   
1      AT  25X-OMVGASSTORA5        OMV Gas Storage GmbH   
2      AT  23X----100225-1C          RAG Energy Storage   
3      AT  37X0000000002964           SEFE Storage GmbH   
4      AT  21X000000001127H  Uniper Energy Storage GmbH   

                                       facility_name      facility_eic  \
0  UGS Haidach (GSA) // historical data prior to ...  25W-SPHAID-GAZ-M   
1  VGS OMV Gas Storage Pool (Tallesbrunn & Schönk...  21W000000000081Y   
2  RAG Storage Pool (Puchkirchen / Haag, Aigelsbr...  21W000000000079L   
3                         UGS Haidach (SEFE Storage)  21W000000000078N   
4          UGS 7 Fields (Uniper Energy Storage) - AT  21W000000000057V   

  facility_type operational_start operational_end  
0           DSR        2011-01-01      2022-10-07  
1           DSR        2011-01-01            None  
2           DSR        

In [4]:
import requests
import pandas as pd
import time

api_key = "e7149a636b4942e8fc7fe6c3d2fbb036"

def fetch_facility_data(facility_eic, country, company_eic, from_date, to_date):
    url = (f"https://agsi.gie.eu/api?country={country}"
           f"&company={company_eic}&facility={facility_eic}"
           f"&from={from_date}&to={to_date}&page=1&size=300")
    
    all_data = []
    headers = {"x-key": api_key}
    
    while url:
        resp = requests.get(url, headers=headers)
        if resp.status_code != 200:
            break
        data = resp.json()
        all_data.extend(data.get("data", []))
        url = data.get("next_page")
        time.sleep(1)
    
    return pd.DataFrame(all_data)

# Example for a single facility
facility_row = facilities_df.iloc[0]
df = fetch_facility_data(
    facility_eic=facility_row["facility_eic"],
    country=facility_row["country"],
    company_eic=facility_row["company_eic"],
    from_date="2011-01-01",
    to_date="2017-12-31"
)
print(df.head())

                                                name              code  \
0  UGS Haidach (GSA) // historical data prior to ...  25W-SPHAID-GAZ-M   
1  UGS Haidach (GSA) // historical data prior to ...  25W-SPHAID-GAZ-M   
2  UGS Haidach (GSA) // historical data prior to ...  25W-SPHAID-GAZ-M   
3  UGS Haidach (GSA) // historical data prior to ...  25W-SPHAID-GAZ-M   
4  UGS Haidach (GSA) // historical data prior to ...  25W-SPHAID-GAZ-M   

                                    url longitude latitude type  \
0  25W-SPHAID-GAZ-M/AT/25X-GSALLC-----E   13.2361  47.9883  DSR   
1  25W-SPHAID-GAZ-M/AT/25X-GSALLC-----E   13.2361  47.9883  DSR   
2  25W-SPHAID-GAZ-M/AT/25X-GSALLC-----E   13.2361  47.9883  DSR   
3  25W-SPHAID-GAZ-M/AT/25X-GSALLC-----E   13.2361  47.9883  DSR   
4  25W-SPHAID-GAZ-M/AT/25X-GSALLC-----E   13.2361  47.9883  DSR   

             updatedAt gasDayStart   gasDayEnd gasInStorage  ...  \
0  2018-01-01 00:00:00  2017-12-31  2018-01-01      14.7432  ...   
1  2017-12-31 00

In [5]:
import requests
import pandas as pd
import time

API_KEY = "e7149a636b4942e8fc7fe6c3d2fbb036"
HEADERS = {"x-key": API_KEY}

def fetch_all_facility_data(facility_eic, country, company_eic, from_date, to_date):
    """Fetch all daily records for one facility over the full date range."""
    all_records = []
    page = 1
    while True:
        url = (f"https://agsi.gie.eu/api?country={country}"
               f"&company={company_eic}&facility={facility_eic}"
               f"&from={from_date}&to={to_date}"
               f"&page={page}&size=300")
        resp = requests.get(url, headers=HEADERS)
        if resp.status_code != 200:
            print(f"Error {resp.status_code} for {facility_eic} page {page}")
            break
        data = resp.json()
        records = data.get("data", [])
        if not records:
            break
        all_records.extend(records)
        # Check if this was the last page
        if page >= data.get("last_page", page):
            break
        page += 1
        time.sleep(0.5)  # polite delay
    return pd.DataFrame(all_records)

# Example: loop over all facilities
all_facility_dfs = []
for idx, row in facilities_df.iterrows():
    print(f"Downloading {row['facility_name']} ({idx+1}/{len(facilities_df)})")
    df = fetch_all_facility_data(
        facility_eic=row["facility_eic"],
        country=row["country"],
        company_eic=row["company_eic"],
        from_date="2011-01-01",
        to_date="2017-12-31"
    )
    if not df.empty:
        df["facility_eic"] = row["facility_eic"]
        df["country"] = row["country"]
        all_facility_dfs.append(df)
    time.sleep(1)  # avoid rate limit

# Combine all
master_df = pd.concat(all_facility_dfs, ignore_index=True)
master_df.to_csv("full_eu_storage_panel.csv", index=False)
print(f"Total rows: {len(master_df)}")

Total rows: 300458


In [6]:
import pandas as pd

storage = pd.read_csv("full_eu_storage_panel.csv")
ttf = pd.read_csv("ttf.csv")

# Convert date columns to datetime
storage["gasDayStart"] = pd.to_datetime(storage["gasDayStart"])
ttf["Date"] = pd.to_datetime(ttf["Date"])

# Merge
df = storage.merge(ttf, left_on="gasDayStart", right_on="Date", how="left")

# Drop rows without price (e.g., holidays)
df = df.dropna(subset=["Price"])

# Optional: create lagged price, seasonal dummies, etc.
df["month"] = df["gasDayStart"].dt.month
df["year"] = df["gasDayStart"].dt.year

FileNotFoundError: [Errno 2] No such file or directory: 'ttf.csv'